In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

from pyspark.sql import functions as F, Window

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_customer
    COMMENT 'Customer dimension — one row per customer. Source: silver.customer'
    AS
    SELECT
        c_custkey    AS customer_key,
        c_name       AS customer_name,
        c_address    AS customer_address,
        c_nationkey  AS nation_key,
        c_phone      AS customer_phone,
        c_acctbal    AS customer_account_balance,
        c_mktsegment AS customer_market_segment
    FROM {catalog}.silver.customer
""")

# readable column comments for downstream consumers (SQL, dashboards, Genie)
for col, comment in {
    "customer_key": "Unique customer identifier (primary key)",
    "customer_name": "Customer name",
    "customer_address": "Customer address",
    "nation_key": "Foreign key to dim_geography (customer geography)",
    "customer_phone": "Customer phone number",
    "customer_account_balance": "Account balance — negative means the customer owes money (valid business data)",
    "customer_market_segment": "Market segment: AUTOMOBILE, BUILDING, FURNITURE, HOUSEHOLD, MACHINERY",
}.items():
    spark.sql(f"ALTER TABLE {catalog}.gold.dim_customer ALTER COLUMN {col} COMMENT '{comment}'")

print(f"dim_customer rows: {spark.table(f'{catalog}.gold.dim_customer').count()}")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_product
    COMMENT 'Product dimension — one row per product. Source: silver.part'
    AS
    SELECT
        p_partkey     AS product_key,
        p_name        AS product_name,
        p_mfgr        AS product_manufacturer,
        p_brand       AS product_brand,
        p_type        AS product_type,
        p_size        AS product_size,
        p_container   AS product_container,
        p_retailprice AS product_retail_price
    FROM {catalog}.silver.part
""")

# readable column comments for downstream consumers (SQL, dashboards, Genie)
for col, comment in {
    "product_key": "Unique product identifier (primary key)",
    "product_name": "Product name",
    "product_manufacturer": "Manufacturer, Manufacturer#1–#5",
    "product_brand": "Product brand (Brand#<manufacturer>#<brand class>)",
    "product_type": "Product type, e.g. ECONOMY ANODIZED STEEL",
    "product_size": "Product size (1–50)",
    "product_container": "Shipping container type (SM CASE, LG BOX, ...)",
    "product_retail_price": "Retail price in USD",
}.items():
    spark.sql(f"ALTER TABLE {catalog}.gold.dim_product ALTER COLUMN {col} COMMENT '{comment}'")

print(f"dim_product rows: {spark.table(f'{catalog}.gold.dim_product').count()}")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_supplier
    COMMENT 'Supplier dimension — one row per supplier, incl. supplier geography (nation + region). Source: silver.supplier + silver.nation + silver.region'
    AS
    SELECT
        s.s_suppkey    AS supplier_key,
        s.s_name       AS supplier_name,
        s.s_address    AS supplier_address,
        s.s_nationkey  AS supplier_nation_key,
        n.n_name       AS supplier_nation_name,
        r.r_name       AS supplier_region_name,
        s.s_phone      AS supplier_phone,
        s.s_acctbal    AS supplier_account_balance
    FROM {catalog}.silver.supplier s
    LEFT JOIN {catalog}.silver.nation n
        ON s.s_nationkey = n.n_nationkey
    LEFT JOIN {catalog}.silver.region r
        ON n.n_regionkey = r.r_regionkey
""")

# readable column comments for downstream consumers (SQL, dashboards, Genie)
for col, comment in {
    "supplier_key": "Unique supplier identifier (primary key)",
    "supplier_name": "Supplier name",
    "supplier_address": "Supplier address",
    "supplier_nation_key": "Nation identifier (FK to dim_geography)",
    "supplier_nation_name": "Nation name of the supplier (denormalized)",
    "supplier_region_name": "Parent region name of the supplier (denormalized)",
    "supplier_phone": "Supplier phone number",
    "supplier_account_balance": "Account balance in USD — negative means the supplier owes money (valid business data)",
}.items():
    spark.sql(f"ALTER TABLE {catalog}.gold.dim_supplier ALTER COLUMN {col} COMMENT '{comment}'")



In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_geography
    COMMENT 'Geography dimension — one row per nation incl. parent region. Source: silver.nation + silver.region'
    AS
    SELECT
        n.n_nationkey AS geography_key,
        n.n_name      AS nation_name,
        n.n_regionkey AS region_key,
        r.r_name      AS region_name
    FROM {catalog}.silver.nation n
    LEFT JOIN {catalog}.silver.region r
        ON n.n_regionkey = r.r_regionkey
""")

# readable column comments for downstream consumers (SQL, dashboards, Genie)
for col, comment in {
    "geography_key": "Unique nation identifier (primary key)",
    "nation_name": "Nation name",
    "region_key": "Parent region identifier",
    "region_name": "Parent region name (denormalized)",
}.items():
    spark.sql(f"ALTER TABLE {catalog}.gold.dim_geography ALTER COLUMN {col} COMMENT '{comment}'")


In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.fact_sales
    COMMENT 'Sales fact — one row per order line. Source: silver.lineitem + silver.order + silver.customer + silver.supplier'
    AS
    SELECT
        li.l_orderkey      AS order_key,
        li.l_linenumber    AS line_number,
        o.o_orderdate      AS order_date,
        o.o_orderstatus    AS order_status,
        o.o_shippriority   AS order_ship_priority,
        o.o_custkey        AS customer_key,
        c.c_nationkey      AS customer_nation_key,
        c.c_mktsegment     AS customer_market_segment,
        li.l_partkey       AS product_key,
        li.l_suppkey       AS supplier_key,
        s.s_nationkey      AS supplier_nation_key,
        li.l_shipdate      AS ship_date,
        li.l_commitdate    AS commit_date,
        li.l_receiptdate   AS receipt_date,
        li.l_shipmode      AS ship_mode,
        li.l_linestatus    AS line_status,
        li.l_quantity      AS quantity,
        li.l_extendedprice AS extended_price,
        li.l_discount      AS discount_rate,
        li.l_tax           AS tax_rate,
        li.l_returnflag    AS return_flag
    FROM {catalog}.silver.lineitem li
    LEFT JOIN {catalog}.silver.order    o ON li.l_orderkey = o.o_orderkey
    LEFT JOIN {catalog}.silver.customer c ON o.o_custkey   = c.c_custkey
    LEFT JOIN {catalog}.silver.supplier s ON li.l_suppkey  = s.s_suppkey
""")

# readable column comments for downstream consumers (SQL, dashboards, Genie)
for col, comment in {
    "order_key": "Order identifier — lines of the same order share it (part of the fact PK)",
    "line_number": "Line number within the order (fact PK = order_key + line_number)",
    "order_date": "Date the order was placed — basis for annual KPIs",
    "order_status": "Order status: O=Open, F=Fulfilled, P=Pending",
    "order_ship_priority": "Ship priority of the order",
    "customer_key": "Customer identifier (FK to dim_customer)",
    "customer_nation_key": "Nation of the ORDERING customer (FK to dim_geography) — customer geography",
    "customer_market_segment": "Market segment of the ordering customer (denormalized)",
    "product_key": "Product identifier (FK to dim_product)",
    "supplier_key": "Supplier identifier (FK to dim_supplier)",
    "supplier_nation_key": "Nation of the SUPPLYING party (FK to dim_geography) — supplier geography",
    "ship_date": "Date the line was shipped",
    "commit_date": "Date the supply was committed",
    "receipt_date": "Date the line was received (always on/after ship_date)",
    "ship_mode": "Shipping mode, e.g. AIR, RAIL, SHIP",
    "line_status": "Line status: O=Outstanding, F=Fulfilled",
    "quantity": "Quantity ordered",
    "extended_price": "Gross revenue measure: quantity times list price",
    "discount_rate": "Discount rate applied (0–0.1)",
    "tax_rate": "Tax rate applied (0–0.08)",
}.items():
    spark.sql(f"ALTER TABLE {catalog}.gold.fact_sales ALTER COLUMN {col} COMMENT '{comment}'")


In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.vw_annual_sales_kpis
    AS
    SELECT
        YEAR(order_date)                                AS order_year,
        SUM(extended_price * (1 - discount_rate))       AS annual_revenue,
        COUNT(DISTINCT order_key)                       AS annual_orders,
        COUNT(DISTINCT customer_key)                    AS active_customers,
        SUM(extended_price) / COUNT(DISTINCT order_key) AS average_order_value
    FROM {catalog}.gold.fact_sales
    GROUP BY YEAR(order_date)
""")

display(spark.sql(f"SELECT * FROM {catalog}.gold.vw_annual_sales_kpis ORDER BY order_year"))

In [0]:
from pyspark.sql import types as T

checks = []

def record(category, check, gold_value, silver_value, ok, action_pass, action_fail):
    checks.append(
        (
            category,
            check,
            int(gold_value),
            int(silver_value),
            int(gold_value) - int(silver_value),
            "PASS" if ok else "FAIL",
            action_pass if ok else action_fail,
        )
    )

# --- fact: row count vs silver.lineitem (grain preserved?) ---
g_rows = spark.table(f"{catalog}.gold.fact_sales").count()
s_rows = spark.table(f"{catalog}.silver.lineitem").count()
record(
    "fact_reconciliation", "Row count: gold.fact_sales vs silver.lineitem",
    g_rows, s_rows, g_rows == s_rows,
    "No action — fact grain preserved (one row per order line)",
    "INVESTIGATE — join fan-out or lost rows between silver.lineitem and gold.fact_sales",
)

# --- fact: gross revenue vs independent Silver calculation ---
g_rev = spark.sql(f"SELECT SUM(extended_price) AS v FROM {catalog}.gold.fact_sales").collect()[0]["v"]
s_rev = spark.sql(f"SELECT SUM(l_extendedprice) AS v FROM {catalog}.silver.lineitem").collect()[0]["v"]
record(
    "fact_reconciliation", "Gross revenue: gold.fact_sales vs silver.lineitem",
    g_rev, s_rev, g_rev == s_rev,
    "No action — Gold matches the independent Silver calculation",
    "INVESTIGATE — measure lost or altered during the Gold build",
)

# --- KPI view: net revenue vs independent Silver calculation ---
g_net = spark.sql(f"SELECT SUM(annual_revenue) AS v FROM {catalog}.gold.vw_annual_sales_kpis").collect()[0]["v"]
s_net = spark.sql(f"SELECT SUM(l_extendedprice * (1 - l_discount)) AS v FROM {catalog}.silver.lineitem").collect()[0]["v"]
record(
    "fact_reconciliation", "Net revenue: vw_annual_sales_kpis vs silver.lineitem",
    g_net, s_net, g_net == s_net,
    "No action — view matches the independent Silver calculation",
    "INVESTIGATE — view aggregation or discount_rate values in the fact",
)

# --- dimensions: row count vs silver source + key uniqueness ---
dims = [
    ("dim_customer",  "customer_key",  "customer", "c_custkey"),
    ("dim_product",   "product_key",   "part",     "p_partkey"),
    ("dim_supplier",  "supplier_key",  "supplier", "s_suppkey"),
    ("dim_geography", "geography_key", "nation",   "n_nationkey"),
]
for dim, key, src, skey in dims:
    d_rows = spark.table(f"{catalog}.gold.{dim}").count()
    d_keys = spark.table(f"{catalog}.gold.{dim}").select(key).distinct().count()
    s_cnt = spark.table(f"{catalog}.silver.{src}").count()
    record(
        "dimension_validation", f"Row count: gold.{dim} vs silver.{src}",
        d_rows, s_cnt, d_rows == s_cnt,
        f"No action — one row per silver.{src} record",
        f"INVESTIGATE — rows lost or duplicated when building gold.{dim}",
    )
    record(
        "dimension_validation", f"Key uniqueness: gold.{dim}.{key}",
        d_rows, d_keys, d_rows == d_keys,
        f"No action — {key} is unique (primary key holds)",
        f"INVESTIGATE — {d_rows - d_keys} duplicate key(s) in gold.{dim}; dimension keys must be unique",
    )

# --- fact grain uniqueness: (order_key, line_number) ---
g_pairs = spark.table(f"{catalog}.gold.fact_sales").select("order_key", "line_number").distinct().count()
record(
    "dimension_validation", "Grain uniqueness: (order_key, line_number) in gold.fact_sales",
    g_rows, g_pairs, g_rows == g_pairs,
    "No action — one row per order line",
    f"INVESTIGATE — {g_rows - g_pairs} duplicate (order_key, line_number) pair(s) in the fact",
)

schema = T.StructType(
    [
        T.StructField("check_category", T.StringType()),
        T.StructField("check_name", T.StringType()),
        T.StructField("gold_value", T.LongType()),
        T.StructField("silver_value", T.LongType()),
        T.StructField("difference", T.LongType()),
        T.StructField("status", T.StringType()),
        T.StructField("action_taken", T.StringType()),
    ]
)

recon_report = spark.createDataFrame(checks, schema).withColumn("generated_at", F.current_timestamp())
recon_report.write.mode("overwrite").saveAsTable(f"{catalog}.gold.reconciliation_report")
spark.sql(f"ALTER TABLE {catalog}.gold.reconciliation_report SET TBLPROPERTIES ('comment' = 'Gold-vs-Silver reconciliation — one row per check with PASS/FAIL status and action taken')")

fails = [c for c in checks if c[5] == "FAIL"]
print(f"Gold reconciliation: {len(checks)} checks -> {len(checks) - len(fails)} PASS, {len(fails)} FAIL")
for fl in fails:
    print("FAIL:", fl[1], "| difference:", fl[4])
print("Report table:", f"{catalog}.gold.reconciliation_report")

display(recon_report.orderBy("check_category", "check_name"))